In [ ]:
# @title 0) Colab 커널 bootstrap: 저장소 clone/update + 최소 설치
import os
import subprocess
import sys
from pathlib import Path

print("[bootstrap] start", flush=True)

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
REPO_REF = os.environ.get("MINDSCOPEX_REPO_REF", "codex/reasoning-output-analysis").strip() or "main"
TARGET = Path("/content/colab")
MARKER = Path("src") / "mindscopex_analysis" / "__init__.py"
CRT_TRANSFORMERS_PROFILE = "ouro"  # 4번 노트북 기본값: Ouro 모델만 확인합니다.
CRT_TRANSFORMERS_PROFILE = os.environ.get("CRT_TRANSFORMERS_PROFILE", CRT_TRANSFORMERS_PROFILE).strip().lower()
# Profiles: qwen35 = install HF transformers main; ouro = pin transformers 4.54.1; none/manual = leave unchanged.
os.environ["CRT_TRANSFORMERS_PROFILE"] = CRT_TRANSFORMERS_PROFILE


def run(cmd, cwd=None, timeout=300, required=True):
    print("+", " ".join(map(str, cmd)), flush=True)
    try:
        subprocess.run(
            cmd,
            cwd=str(cwd) if cwd else None,
            timeout=timeout,
            check=True,
        )
    except Exception as exc:
        print(f"[bootstrap] command failed: {exc}", flush=True)
        if required:
            raise


def repair_pillow_stack():
    # Colab can keep mismatched Pillow files after pip upgrades, which breaks transformers -> torchvision -> PIL imports.
    run([sys.executable, "-m", "pip", "uninstall", "-y", "Pillow", "pillow-simd"], timeout=300, required=False)
    run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "Pillow>=10.4,<12"], timeout=600)


def require_restart_if_loaded(*prefixes):
    loaded = [m for m in sys.modules if any(m == p or m.startswith(p + ".") for p in prefixes)]
    if loaded:
        raise SystemExit(
            "Pillow/torchvision/transformers was already imported in this runtime. "
            "Restart the Colab runtime, then rerun from the first bootstrap cell."
        )


if Path("/content").exists():
    if (TARGET / ".git").exists():
        run(["git", "fetch", "origin", REPO_REF], cwd=TARGET, timeout=90, required=False)
        run(["git", "checkout", "-B", REPO_REF, f"origin/{REPO_REF}"], cwd=TARGET, timeout=60, required=False)
        run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=TARGET, timeout=90, required=False)
    elif not TARGET.exists():
        run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(TARGET)], timeout=180)
    elif not (TARGET / MARKER).is_file():
        alt = Path("/content/mindscopex_analysis")
        if (alt / ".git").exists():
            run(["git", "fetch", "origin", REPO_REF], cwd=alt, timeout=90, required=False)
            run(["git", "checkout", "-B", REPO_REF, f"origin/{REPO_REF}"], cwd=alt, timeout=60, required=False)
            run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=alt, timeout=90, required=False)
        elif not alt.exists():
            run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(alt)], timeout=180)
        TARGET = alt
    if not (TARGET / MARKER).is_file():
        raise FileNotFoundError(f"저장소 marker를 찾지 못했습니다: {TARGET / MARKER}")
    os.chdir(TARGET)
    os.environ["MINDSCOPEX_ROOT"] = str(TARGET)
    print("cwd =", Path.cwd(), flush=True)
    print("REPO_REF =", REPO_REF, flush=True)
    print("MINDSCOPEX_ROOT =", os.environ["MINDSCOPEX_ROOT"], flush=True)
    run([sys.executable, "-m", "pip", "install", "-e", "."], timeout=600)
    print("CRT_TRANSFORMERS_PROFILE =", CRT_TRANSFORMERS_PROFILE, flush=True)
    if CRT_TRANSFORMERS_PROFILE == "qwen35":
        run([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-U",
            "transformers @ git+https://github.com/huggingface/transformers.git@main",
            "accelerate",
        ], timeout=1200)
        repair_pillow_stack()
    elif CRT_TRANSFORMERS_PROFILE == "ouro":
        run([sys.executable, "-m", "pip", "install", "transformers==4.54.1", "accelerate"], timeout=600)
        repair_pillow_stack()
    elif CRT_TRANSFORMERS_PROFILE in {"none", "manual"}:
        print("[bootstrap] leaving transformers unchanged", flush=True)
        repair_pillow_stack()
    else:
        raise ValueError("Unknown CRT_TRANSFORMERS_PROFILE: " + CRT_TRANSFORMERS_PROFILE)
    require_restart_if_loaded("PIL", "torchvision", "transformers")
    print("[bootstrap] done", flush=True)
else:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / MARKER).is_file():
            os.environ["MINDSCOPEX_ROOT"] = str(base)
            print("로컬 저장소에서 실행 중입니다:", base, flush=True)
            break
    else:
        print("저장소 루트를 찾지 못했습니다. Colab 커널이면 이 셀을 맨 먼저 다시 실행하세요.", flush=True)

# CRT bat-and-ball lure feature MVP

목표는 전체 RQ1을 돌리기 전에, bat-and-ball CRT에서 아래 패턴을 보이는 SAE feature가 실제로 있는지 최소 실험으로 확인하는 것입니다.

| 입력/상황 | 기대 activation |
|---|---|
| bat-and-ball 문제에서 답을 내기 직전 | 높음 |
| 모델이 `10 cents`라고 틀리게 답하는 run | 매우 높음 |
| 모델이 `5 cents`라고 맞히는 run의 초기 단계 | 잠깐 높음 |
| 모델이 식을 세우고 검산하는 후반 단계 | 낮아짐 |
| 단순히 `10 cents`가 나오는 일반 문장 | 낮거나 중간 |
| 함정이 없는 돈 계산 문제 | 낮음 |

이 노트북은 실제 generation을 먼저 신뢰하지 않고, 위 상황을 forced transcript / probe context로 만들어 Qwen3-Base residual stream을 Qwen-Scope SAE에 투영합니다. 발견은 일부 paraphrase에서 하고, 검증은 남겨둔 paraphrase에서 봅니다.


## 1. Import와 설정


In [ ]:
import gc
import os
import re
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

try:
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go
    import plotly.io as pio
    import torch
    from tqdm.auto import tqdm
    from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast, set_seed
    import transformers
except (ModuleNotFoundError, ImportError) as exc:
    missing = getattr(exc, "name", None) or str(exc)
    if "PIL._typing" in str(exc) or "_Ink" in str(exc):
        raise ImportError(
            "Pillow/torchvision import state is inconsistent in this runtime. "
            "Colab이면 0번 bootstrap 셀을 다시 실행한 뒤 런타임을 재시작하세요. "
            "이 노트북은 이제 AutoProcessor/torchvision을 top-level에서 import하지 않습니다."
        ) from exc
    raise ModuleNotFoundError(
        f"Missing notebook dependency: {missing}. "
        "Colab이면 맨 위 bootstrap 셀을 먼저 실행하고 런타임을 재시작하세요. "
        "로컬이면 저장소 루트에서 `uv sync` 또는 `python -m pip install -e .`를 실행하세요."
    ) from exc
_REPO_MARK = Path("src") / "mindscopex_analysis" / "__init__.py"


def _find_repo_root() -> Path:
    env = os.environ.get("MINDSCOPEX_ROOT", "").strip()
    if env:
        root = Path(env).expanduser().resolve()
        if (root / _REPO_MARK).is_file():
            return root
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / _REPO_MARK).is_file():
            return base
    raise FileNotFoundError("src/mindscopex_analysis 를 찾지 못했습니다.")


ROOT = _find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from mindscopex_analysis.notebook_utils import dtype_from_str
    from mindscopex_analysis.qwen_scope import (
        capture_residuals,
        format_qwen_chat,
        get_transformer_layers,
        load_qwen_scope_sae,
        summarize_qwen_scope_features,
    )
except ModuleNotFoundError as exc:
    missing = exc.name or str(exc)
    raise ModuleNotFoundError(
        f"Missing project dependency while importing mindscopex_analysis: {missing}. "
        "Colab이면 맨 위 bootstrap 셀을 먼저 실행하고 런타임을 재시작하세요. "
        "로컬이면 저장소 루트에서 `uv sync` 또는 `python -m pip install -e .`를 실행하세요."
    ) from exc

pio.renderers.default = "plotly_mimetype"
set_seed(42)
print("ROOT =", ROOT)
print("torch =", torch.__version__, "cuda =", torch.cuda.is_available())
print("transformers =", transformers.__version__)

In [ ]:
MODEL_ID = "Qwen/Qwen3-1.7B-Base"
SAE_REPO = "Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_50"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = dtype_from_str("bfloat16" if DEVICE == "cuda" else "float32")
LAYERS = [6, 14, 21, 27]
SAE_TOP_K = 50
BATCH_SIZE = 64
MAX_LENGTH = 1024
TOKEN_POSITION = "last"
DISCOVERY_TOP_N = 40
REPORT_TOP_N = 20

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
OUT_DIR = ROOT / "outputs" / "crt_lure_feature_mvp" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"model={MODEL_ID}")
print(f"sae={SAE_REPO}")
print(f"device={DEVICE} dtype={DTYPE} layers={LAYERS}")
print("out:", OUT_DIR)


## 2. Ouro 모델 직접답변 정성 확인

SAE feature를 보기 전에, 기본 실행에서는 Ouro 모델군이 bat-and-ball 문제에서 실제로 `10 cents` lure에 빠지는지 먼저 봅니다. 이전 셀처럼 `Solve carefully`, `step by step`을 유도하지 않고, 모든 prompt는 **최종 금액만** 답하도록 제한합니다.

`non_think`/`think`는 chat template의 thinking switch와 프롬프트를 함께 분리합니다. `non_think`는 직접 답변 전용으로 짧은 생성 예산을 쓰고 `<think>` 토큰을 금지해 reasoning trace가 답변 분석에 섞이지 않게 합니다. 그래도 모델이 reasoning을 출력하면 `unexpected_thinking_trace`/`leaked_thinking_words`로 따로 표시합니다.

`think`는 더 긴 생성 예산을 쓰고, raw 출력의 `<think>...</think>` 구간은 `thinking`에, 이후 최종 답변은 `answer`에 저장합니다. `<think>`가 닫히지 않으면 `truncated_thinking=True`, `no_final_answer`로 세어 최종 답과 reasoning을 섞어 분류하지 않습니다.

Looped transformer 계열은 Parcae와 Ouro family를 추가합니다. Parcae는 140M/370M/770M/1.3B 사이즈가 공개된 stable looped LM family이고, instruction/chat thinking switch가 없는 base LM이라 결과표에서는 `think=plain`으로 비교합니다. Ouro는 1.4B/2.6B 및 Thinking 체크포인트를 `think=ouro_plain`/`think=ouro_thinking`으로 비교합니다. 아래 feature MVP는 여전히 Base 모델 + Qwen-Scope SAE로 진행합니다.

In [ ]:
RUN_QUALITATIVE_GENERATION = True
RUNTIME_TRANSFORMERS_PROFILE = os.environ.get("CRT_TRANSFORMERS_PROFILE", "ouro").strip().lower()
QUAL_N_SAMPLES = 3
QUAL_MAX_NEW_TOKENS_THINK_OFF = 16
QUAL_MAX_NEW_TOKENS_THINK_ON = 2048
QUAL_MAX_NEW_TOKENS_LOOPED = 64
QUAL_MAX_NEW_TOKENS_OURO = 128
QUAL_RUN_ONLY_FIRST_N_MODELS = None  # smoke test는 1 또는 2로 지정
QUAL_FAMILIES_TO_RUN = ["ouro"]
# 직접 관리한 런타임에서 전부 돌리고 싶으면 None, 특정 family만 돌리려면 예: ["ouro"]
RUN_LOOPED_QUALITATIVE_MODELS = True
RUN_OURO_QUALITATIVE_MODELS = True
INSTALL_PARCAE_LM = True
PARCAE_TOKENIZER_ID = "SandyResearch/parcae-tokenizer"

NON_THINK_SYSTEM_PROMPT = (
    "You are in direct-answer mode. Return exactly one final amount immediately. "
    "Do not write reasoning, derivations, equations, or explanations."
)
THINK_SYSTEM_PROMPT = (
    "Use the model's thinking section for any reasoning. After the thinking section, "
    "return exactly one final amount and no explanation."
)
NON_THINK_USER_SUFFIX = "\nDirect-answer mode: output exactly one amount only. Do not include reasoning."
THINK_USER_SUFFIX = "\nThinking mode: reason only inside the thinking section, then output exactly one amount only."
QWEN_THINK_TOKENS_TO_BLOCK = ["<think>", "</think>"]

QUAL_MODEL_SPECS = [
    {"tag": "qwen35_2b", "family": "qwen35", "runner": "qwen35_processor", "model_id": "Qwen/Qwen3.5-2B", "enabled": True},
    {"tag": "qwen35_9b", "family": "qwen35", "runner": "qwen35_processor", "model_id": "Qwen/Qwen3.5-9B", "enabled": True},
    {"tag": "qwen35_27b", "family": "qwen35", "runner": "qwen35_processor", "model_id": "Qwen/Qwen3.5-27B", "enabled": True},
    {"tag": "qwen35_35b_a3b", "family": "qwen35", "runner": "qwen35_processor", "model_id": "Qwen/Qwen3.5-35B-A3B", "enabled": True},
    {"tag": "parcae_140m", "family": "parcae", "runner": "parcae", "model_id": "SandyResearch/parcae-140m", "enabled": True},
    {"tag": "parcae_370m", "family": "parcae", "runner": "parcae", "model_id": "SandyResearch/parcae-370m", "enabled": True},
    {"tag": "parcae_770m", "family": "parcae", "runner": "parcae", "model_id": "SandyResearch/parcae-770m", "enabled": True},
    {"tag": "parcae_1p3b", "family": "parcae", "runner": "parcae", "model_id": "SandyResearch/parcae-1.3b", "enabled": True},
    {"tag": "ouro_1p4b", "family": "ouro", "runner": "ouro_causal_lm", "model_id": "ByteDance/Ouro-1.4B", "think": "ouro_plain", "enabled": True},
    {"tag": "ouro_2p6b", "family": "ouro", "runner": "ouro_causal_lm", "model_id": "ByteDance/Ouro-2.6B", "think": "ouro_plain", "enabled": True},
    {"tag": "ouro_1p4b_thinking", "family": "ouro", "runner": "ouro_causal_lm", "model_id": "ByteDance/Ouro-1.4B-Thinking", "think": "ouro_thinking", "enabled": True},
    {"tag": "ouro_2p6b_thinking", "family": "ouro", "runner": "ouro_causal_lm", "model_id": "ByteDance/Ouro-2.6B-Thinking", "think": "ouro_thinking", "enabled": True},
]

BAT_BALL_QUESTION = (
    "A bat and a ball cost $1.10 in total. "
    "The bat costs $1.00 more than the ball. "
    "How much does the ball cost?"
)

QUAL_PROMPTS = [
    {
        "prompt_id": "amount_only_direct",
        "prompt": BAT_BALL_QUESTION + " Reply with exactly one amount only. Do not explain.",
    },
    {
        "prompt_id": "amount_only_immediate",
        "prompt": "Answer immediately with exactly one amount only, no explanation: " + BAT_BALL_QUESTION,
    },
]

QUAL_RUNS = [
    {"mode": "non_think_greedy", "think": "non_think", "enable_thinking": False, "allow_thinking_trace": False, "decode": "greedy", "do_sample": False, "temperature": None, "top_p": None, "n": 1},
    {"mode": "non_think_sample", "think": "non_think", "enable_thinking": False, "allow_thinking_trace": False, "decode": "sample", "do_sample": True, "temperature": 0.7, "top_p": 0.9, "n": QUAL_N_SAMPLES},
    {"mode": "think_greedy", "think": "think", "enable_thinking": True, "allow_thinking_trace": True, "decode": "greedy", "do_sample": False, "temperature": None, "top_p": None, "n": 1},
    {"mode": "think_sample", "think": "think", "enable_thinking": True, "allow_thinking_trace": True, "decode": "sample", "do_sample": True, "temperature": 0.7, "top_p": 0.9, "n": QUAL_N_SAMPLES},
]
LOOPED_QUAL_RUNS = [
    {"mode": "plain_greedy", "think": "plain", "decode": "greedy", "do_sample": False, "temperature": None, "top_p": None, "n": 1, "max_new_tokens": QUAL_MAX_NEW_TOKENS_LOOPED},
    {"mode": "plain_sample", "think": "plain", "decode": "sample", "do_sample": True, "temperature": 0.7, "top_p": 0.9, "n": QUAL_N_SAMPLES, "max_new_tokens": QUAL_MAX_NEW_TOKENS_LOOPED},
]
OURO_QUAL_RUNS = [
    {"mode": "ouro_greedy", "decode": "greedy", "do_sample": False, "temperature": None, "top_p": None, "n": 1, "max_new_tokens": QUAL_MAX_NEW_TOKENS_OURO},
    {"mode": "ouro_sample", "decode": "sample", "do_sample": True, "temperature": 0.7, "top_p": 0.9, "n": QUAL_N_SAMPLES, "max_new_tokens": QUAL_MAX_NEW_TOKENS_OURO},
]

QUAL_RESULT_COLUMNS = [
    "family", "runner", "tag", "model_id", "think", "mode", "decode", "prompt_id", "sample_idx", "label",
    "is_correct", "enable_thinking", "allow_thinking_trace", "answer_only", "has_final_answer",
    "has_thinking_trace", "unexpected_thinking_trace", "truncated_thinking", "answer_source", "answer_words",
    "thinking_words", "leaked_thinking_words", "generated_tokens", "max_new_tokens", "answer", "thinking",
    "thinking_excerpt", "leaked_thinking", "leaked_thinking_excerpt", "raw_generation",
]
QUAL_FAILURE_COLUMNS = ["family", "runner", "tag", "model_id", "error_type", "error"]


def classify_bat_ball_answer(text: str) -> str:
    t = text.lower()
    has_5 = bool(re.search(r"(?<![0-9])5(?![0-9])\s*cents?|\$\s*0\.05\b|0\.05\s*dollars?", t))
    has_10 = bool(re.search(r"(?<![0-9])10(?![0-9])\s*cents?|\$\s*0\.10\b|0\.10\s*dollars?", t))
    if has_5 and not has_10:
        return "correct_5"
    if has_10 and not has_5:
        return "lure_10"
    if has_5 and has_10:
        return "mentions_both"
    return "other"


def is_answer_only(text: str) -> bool:
    cleaned = re.sub(r"\s+", " ", text.strip())
    if not cleaned:
        return False
    words = re.findall(r"\S+", cleaned)
    explanation_cues = [
        "because", "since", "therefore", "let ", "equation", "total", "more than",
        "so ", "=", "first", "subtract", "divide", "costs $1", "costs one",
    ]
    return len(words) <= 8 and not any(cue in cleaned.lower() for cue in explanation_cues)


def _clean_generation_text(text: str) -> str:
    part = "" if text is None else str(text)
    for tok in (
        "<|im_start|>",
        "<|im_end|>",
        "<|endoftext|>",
        "<|end_of_text|>",
    ):
        part = part.replace(tok, "")
    return part.strip()


def _extract_qwen_think_parts(raw: str) -> dict:
    text = _clean_generation_text(raw)
    start = text.find("<think>")
    end = text.find("</think>")
    has_open = start != -1
    has_close = has_open and end != -1 and end > start
    truncated = has_open and not has_close
    if not has_open:
        return {
            "raw_clean": text,
            "thinking": "",
            "answer_after_think": text,
            "outside_think": text,
            "has_thinking_trace": False,
            "truncated_thinking": False,
        }

    before = text[:start].strip()
    if has_close:
        thinking = text[start + len("<think>") : end].strip()
        after = text[end + len("</think>") :].strip()
        outside = "\n".join(part for part in (before, after) if part)
    else:
        thinking = text[start + len("<think>") :].strip()
        after = ""
        outside = before
    return {
        "raw_clean": text,
        "thinking": thinking,
        "answer_after_think": after,
        "outside_think": outside,
        "has_thinking_trace": True,
        "truncated_thinking": truncated,
    }


def _allow_thinking_trace(run_cfg: dict) -> bool:
    return bool(run_cfg.get("allow_thinking_trace", run_cfg.get("enable_thinking", False)))


def parse_qwen_generation(raw: str, *, allow_thinking_trace: bool) -> dict:
    parts = _extract_qwen_think_parts(raw)
    thinking = parts["thinking"] if allow_thinking_trace else ""
    leaked_thinking = ""
    answer_source = "raw_direct"

    if allow_thinking_trace:
        if parts["truncated_thinking"]:
            visible_answer = ""
            answer_source = "truncated_think"
        elif parts["has_thinking_trace"]:
            visible_answer = parts["answer_after_think"].strip()
            answer_source = "after_think" if visible_answer else "missing_after_think"
        else:
            visible_answer = parts["raw_clean"]
            answer_source = "raw_no_think_tag"
    else:
        visible_answer = parts["outside_think"].strip() if parts["has_thinking_trace"] else parts["raw_clean"]
        answer_source = "outside_unexpected_think" if parts["has_thinking_trace"] else "raw_direct"
        if parts["has_thinking_trace"]:
            leaked_thinking = parts["thinking"]
        elif visible_answer and not is_answer_only(visible_answer):
            leaked_thinking = visible_answer

    has_final_answer = bool(visible_answer)
    unexpected_thinking_trace = (not allow_thinking_trace) and (
        parts["has_thinking_trace"] or (has_final_answer and not is_answer_only(visible_answer))
    )
    return {
        "thinking": thinking,
        "visible_answer": visible_answer,
        "has_final_answer": has_final_answer,
        "has_thinking_trace": parts["has_thinking_trace"],
        "unexpected_thinking_trace": unexpected_thinking_trace,
        "truncated_thinking": parts["truncated_thinking"],
        "answer_source": answer_source,
        "leaked_thinking": leaked_thinking,
    }


def max_new_tokens_for_run(run_cfg: dict) -> int:
    if "max_new_tokens" in run_cfg:
        return int(run_cfg["max_new_tokens"])
    return QUAL_MAX_NEW_TOKENS_THINK_ON if run_cfg.get("enable_thinking") else QUAL_MAX_NEW_TOKENS_THINK_OFF


def runs_for_spec(spec: dict) -> list[dict]:
    if spec.get("runner") == "parcae":
        return LOOPED_QUAL_RUNS
    if spec.get("runner") == "ouro_causal_lm":
        return OURO_QUAL_RUNS
    return QUAL_RUNS


def selected_qual_specs() -> list[dict]:
    specs = [s for s in QUAL_MODEL_SPECS if s.get("enabled", False)]
    if QUAL_FAMILIES_TO_RUN is not None:
        allowed_families = set(QUAL_FAMILIES_TO_RUN)
        specs = [s for s in specs if s.get("family") in allowed_families]
    if not RUN_LOOPED_QUALITATIVE_MODELS:
        specs = [s for s in specs if s.get("family") != "parcae"]
    if not RUN_OURO_QUALITATIVE_MODELS:
        specs = [s for s in specs if s.get("family") != "ouro"]
    if QUAL_RUN_ONLY_FIRST_N_MODELS is not None:
        specs = specs[: int(QUAL_RUN_ONLY_FIRST_N_MODELS)]
    return specs


def transformers_has_arch(model_type: str) -> bool:
    try:
        from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    except Exception:
        return False
    return model_type in CONFIG_MAPPING_NAMES


print("runtime transformers profile =", RUNTIME_TRANSFORMERS_PROFILE)
print("qwen3_5 arch support =", transformers_has_arch("qwen3_5"))
SELECTED_QUAL_SPECS = selected_qual_specs()
if any(s.get("runner") == "qwen35_processor" for s in SELECTED_QUAL_SPECS):
    if not transformers_has_arch("qwen3_5"):
        raise RuntimeError(
            "This transformers build does not support model_type=qwen3_5. "
            "In Colab, rerun the first bootstrap cell and restart the runtime if transformers was already imported."
        )
if any(s.get("runner") == "ouro_causal_lm" for s in SELECTED_QUAL_SPECS):
    from packaging.version import Version
    if Version(transformers.__version__) >= Version("4.56.0"):
        raise RuntimeError(
            "Ouro model card recommends transformers<4.56.0. "
            "Set CRT_TRANSFORMERS_PROFILE=ouro in the first bootstrap cell, restart the runtime, then rerun."
        )

pd.DataFrame(SELECTED_QUAL_SPECS).pipe(display)
pd.DataFrame(QUAL_PROMPTS).pipe(display)

In [ ]:
def _qual_model_kwargs() -> dict:
    kwargs = {
        "torch_dtype": DTYPE,
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
    }
    if DEVICE == "cuda":
        kwargs["device_map"] = "auto"
    return kwargs


def _move_qual_inputs(inputs):
    if hasattr(inputs, "to"):
        return inputs.to(DEVICE)
    return {k: v.to(DEVICE) if hasattr(v, "to") else v for k, v in inputs.items()}


def _qwen35_content(text: str) -> str:
    return text


def _qwen35_messages(
    prompt: str,
    *,
    enable_thinking: bool | None = None,
    thinking_fallback: bool | None = None,
) -> list[dict]:
    use_thinking_prompt = bool(enable_thinking if enable_thinking is not None else thinking_fallback)
    system_prompt = THINK_SYSTEM_PROMPT if use_thinking_prompt else NON_THINK_SYSTEM_PROMPT
    user_suffix = THINK_USER_SUFFIX if use_thinking_prompt else NON_THINK_USER_SUFFIX
    control_suffix = ""
    if thinking_fallback is True:
        control_suffix = "\n/think"
    elif thinking_fallback is False:
        control_suffix = "\n/no_think"
    return [
        {"role": "system", "content": _qwen35_content(system_prompt)},
        {"role": "user", "content": _qwen35_content(prompt + user_suffix + control_suffix)},
    ]


def _processor_inputs(processor, prompt: str, enable_thinking: bool):
    kwargs = {
        "add_generation_prompt": True,
        "tokenize": True,
        "return_dict": True,
        "return_tensors": "pt",
    }
    try:
        return processor.apply_chat_template(
            _qwen35_messages(prompt, enable_thinking=enable_thinking),
            enable_thinking=enable_thinking,
            **kwargs,
        )
    except TypeError:
        messages = _qwen35_messages(
            prompt,
            enable_thinking=enable_thinking,
            thinking_fallback=enable_thinking,
        )
        try:
            return processor.apply_chat_template(messages, **kwargs)
        except TypeError:
            chat_text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
            return processor(text=[chat_text], return_tensors="pt")


def _decode_processor(processor, token_ids) -> str:
    decoder = processor if hasattr(processor, "decode") else getattr(processor, "tokenizer")
    try:
        return decoder.decode(token_ids, skip_special_tokens=False).strip()
    except TypeError:
        return decoder.decode(token_ids).strip()


def _bad_words_ids_for_texts(tokenizer, texts: list[str]) -> list[list[int]]:
    if tokenizer is None:
        return []
    bad_words = []
    for text in texts:
        try:
            ids = tokenizer.encode(text, add_special_tokens=False)
        except Exception:
            continue
        if ids:
            bad_words.append(ids)
    return bad_words


def _apply_generation_guards(gen_kwargs: dict, tokenizer, run_cfg: dict) -> None:
    if _allow_thinking_trace(run_cfg):
        return
    bad_words_ids = _bad_words_ids_for_texts(tokenizer, QWEN_THINK_TOKENS_TO_BLOCK)
    if bad_words_ids:
        gen_kwargs["bad_words_ids"] = bad_words_ids


def _qual_row(spec: dict, run_cfg: dict, prompt_spec: dict, sample_idx: int, raw: str, generated_tokens: int, max_new_tokens: int) -> dict:
    allow_thinking_trace = _allow_thinking_trace(run_cfg)
    if spec.get("runner") == "qwen35_processor":
        parsed = parse_qwen_generation(raw, allow_thinking_trace=allow_thinking_trace)
    else:
        visible_answer = _clean_generation_text(raw)
        parsed = {
            "thinking": "",
            "visible_answer": visible_answer,
            "has_final_answer": bool(visible_answer),
            "has_thinking_trace": False,
            "unexpected_thinking_trace": False,
            "truncated_thinking": False,
            "answer_source": "raw_direct",
            "leaked_thinking": "",
        }
    visible_answer = parsed["visible_answer"]
    thinking = parsed["thinking"]
    leaked_thinking = parsed["leaked_thinking"]
    label = "no_final_answer" if not parsed["has_final_answer"] else classify_bat_ball_answer(visible_answer)
    return {
        "family": spec.get("family", "unknown"),
        "runner": spec.get("runner", "qwen35_processor"),
        "tag": spec["tag"],
        "model_id": spec["model_id"],
        "think": spec.get("think", run_cfg.get("think", "think" if run_cfg.get("enable_thinking") else "non_think")),
        "mode": run_cfg["mode"],
        "decode": run_cfg["decode"],
        "prompt_id": prompt_spec["prompt_id"],
        "sample_idx": sample_idx,
        "label": label,
        "is_correct": label == "correct_5",
        "enable_thinking": run_cfg.get("enable_thinking"),
        "allow_thinking_trace": allow_thinking_trace,
        "answer_only": is_answer_only(visible_answer),
        "has_final_answer": parsed["has_final_answer"],
        "has_thinking_trace": parsed["has_thinking_trace"],
        "unexpected_thinking_trace": parsed["unexpected_thinking_trace"],
        "truncated_thinking": parsed["truncated_thinking"],
        "answer_source": parsed["answer_source"],
        "answer_words": len(visible_answer.split()),
        "thinking_words": len(thinking.split()),
        "leaked_thinking_words": len(leaked_thinking.split()),
        "generated_tokens": int(generated_tokens),
        "max_new_tokens": int(max_new_tokens),
        "answer": visible_answer,
        "thinking": thinking,
        "thinking_excerpt": thinking[:600],
        "leaked_thinking": leaked_thinking,
        "leaked_thinking_excerpt": leaked_thinking[:600],
        "raw_generation": raw,
    }


def _generate_qwen35_model(spec: dict) -> list[dict]:
    tag = spec["tag"]
    model_id = spec["model_id"]
    print(f"\n===== loading {tag}: {model_id} =====", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, **_qual_model_kwargs()).eval()
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    rows = []
    try:
        for run_cfg in runs_for_spec(spec):
            for prompt_spec in QUAL_PROMPTS:
                reps = int(run_cfg["n"])
                for sample_idx in range(reps):
                    inputs = _processor_inputs(tokenizer, prompt_spec["prompt"], bool(run_cfg["enable_thinking"]))
                    inputs = _move_qual_inputs(inputs)
                    input_len = inputs["input_ids"].shape[-1]
                    max_new_tokens = max_new_tokens_for_run(run_cfg)
                    gen_kwargs = {
                        "max_new_tokens": max_new_tokens,
                        "do_sample": bool(run_cfg["do_sample"]),
                    }
                    if tokenizer.pad_token_id is not None:
                        gen_kwargs["pad_token_id"] = tokenizer.pad_token_id
                    if tokenizer.eos_token_id is not None:
                        gen_kwargs["eos_token_id"] = tokenizer.eos_token_id
                    _apply_generation_guards(gen_kwargs, tokenizer, run_cfg)
                    if run_cfg["do_sample"]:
                        gen_kwargs.update({
                            "temperature": float(run_cfg["temperature"]),
                            "top_p": float(run_cfg["top_p"]),
                            "top_k": 20,
                        })
                    with torch.no_grad():
                        out = model.generate(**inputs, **gen_kwargs)
                    new_tokens = out[0, input_len:]
                    raw = _decode_processor(tokenizer, new_tokens)
                    rows.append(_qual_row(spec, run_cfg, prompt_spec, sample_idx, raw, int(new_tokens.numel()), max_new_tokens))
    finally:
        del model, tokenizer
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return rows


def _ensure_parcae_lm():
    try:
        import parcae_lm
        return parcae_lm
    except ImportError:
        if not INSTALL_PARCAE_LM:
            raise RuntimeError("parcae-lm is not installed. Set INSTALL_PARCAE_LM=True or install it manually.")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/sandyresearch/parcae.git"])
        import parcae_lm
        return parcae_lm


def _format_looped_prompt(prompt: str) -> str:
    return (
        "Question:\n"
        f"{prompt}\n\n"
        "Answer with exactly one amount only. Do not explain.\n"
        "Answer:"
    )


def _format_ouro_prompt(tokenizer, prompt: str) -> str:
    messages = [
        {"role": "system", "content": "Return exactly one final amount. Do not show explanation or derivation."},
        {"role": "user", "content": prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return _format_looped_prompt(prompt)


def _filter_top_p(logits: torch.Tensor, top_p: float) -> torch.Tensor:
    if top_p is None or top_p >= 1.0:
        return logits
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    probs = torch.softmax(sorted_logits, dim=-1)
    cumulative = torch.cumsum(probs, dim=-1)
    remove = cumulative > float(top_p)
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    filtered = logits.clone()
    filtered.scatter_(dim=-1, index=sorted_indices, src=sorted_logits.masked_fill(remove, -float("inf")))
    return filtered


def _sample_next_token(logits: torch.Tensor, run_cfg: dict) -> torch.Tensor:
    if not run_cfg["do_sample"]:
        return torch.argmax(logits, dim=-1, keepdim=True)
    logits = logits / float(run_cfg.get("temperature") or 1.0)
    top_k = int(run_cfg.get("top_k", 20))
    if top_k > 0 and top_k < logits.shape[-1]:
        kth = torch.topk(logits, top_k, dim=-1).values[..., -1, None]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    logits = _filter_top_p(logits, float(run_cfg.get("top_p") or 1.0))
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)


def _generate_parcae_tokens(model, tokenizer, prompt: str, run_cfg: dict) -> tuple[str, int, int]:
    encoded = tokenizer(_format_looped_prompt(prompt), return_tensors="pt")
    generated = encoded["input_ids"].to(DEVICE)
    input_len = generated.shape[-1]
    eos_token_id = tokenizer.eos_token_id
    max_new_tokens = max_new_tokens_for_run(run_cfg)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            attention_mask = torch.ones_like(generated, device=generated.device)
            out = model(generated, attention_mask=attention_mask, return_logits=True)
            logits = out["logits"][:, -1, :]
            next_token = _sample_next_token(logits, run_cfg)
            generated = torch.cat([generated, next_token.to(generated.device)], dim=-1)
            if eos_token_id is not None and int(next_token.item()) == int(eos_token_id):
                break
    new_tokens = generated[0, input_len:]
    raw = tokenizer.decode(new_tokens.tolist(), skip_special_tokens=True).strip()
    return raw, int(new_tokens.numel()), max_new_tokens


def _generate_parcae_model(spec: dict) -> list[dict]:
    if not RUN_LOOPED_QUALITATIVE_MODELS:
        return []
    parcae_lm = _ensure_parcae_lm()
    tag = spec["tag"]
    model_id = spec["model_id"]
    print(f"\n===== loading {tag}: {model_id} (parcae looped) =====", flush=True)
    tokenizer = PreTrainedTokenizerFast.from_pretrained(PARCAE_TOKENIZER_ID)
    model_kwargs = {"device": DEVICE}
    if DEVICE == "cuda":
        model_kwargs["dtype"] = DTYPE
    model = parcae_lm.from_pretrained(model_id, **model_kwargs).eval()
    rows = []
    try:
        for run_cfg in runs_for_spec(spec):
            for prompt_spec in QUAL_PROMPTS:
                for sample_idx in range(int(run_cfg["n"])):
                    raw, generated_tokens, max_new_tokens = _generate_parcae_tokens(model, tokenizer, prompt_spec["prompt"], run_cfg)
                    rows.append(_qual_row(spec, run_cfg, prompt_spec, sample_idx, raw, generated_tokens, max_new_tokens))
    finally:
        del model, tokenizer
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return rows


def _generate_ouro_model(spec: dict) -> list[dict]:
    if not RUN_OURO_QUALITATIVE_MODELS:
        return []
    tag = spec["tag"]
    model_id = spec["model_id"]
    print(f"\n===== loading {tag}: {model_id} (ouro causal lm) =====", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, **_qual_model_kwargs()).eval()
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token
    rows = []
    try:
        for run_cfg in runs_for_spec(spec):
            for prompt_spec in QUAL_PROMPTS:
                formatted = _format_ouro_prompt(tokenizer, prompt_spec["prompt"])
                inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
                inputs = _move_qual_inputs(inputs)
                input_len = inputs["input_ids"].shape[-1]
                for sample_idx in range(int(run_cfg["n"])):
                    max_new_tokens = max_new_tokens_for_run(run_cfg)
                    generated = inputs["input_ids"]
                    eos_token_id = tokenizer.eos_token_id
                    with torch.no_grad():
                        for _ in range(max_new_tokens):
                            attention_mask = torch.ones_like(generated, device=generated.device)
                            out = model(input_ids=generated, attention_mask=attention_mask, use_cache=False)
                            logits = out.logits[:, -1, :]
                            next_token = _sample_next_token(logits, run_cfg)
                            generated = torch.cat([generated, next_token.to(generated.device)], dim=-1)
                            if eos_token_id is not None and int(next_token.item()) == int(eos_token_id):
                                break
                    new_tokens = generated[0, input_len:]
                    raw = tokenizer.decode(new_tokens.tolist(), skip_special_tokens=True).strip()
                    rows.append(_qual_row(spec, run_cfg, prompt_spec, sample_idx, raw, int(new_tokens.numel()), max_new_tokens))
    finally:
        del model, tokenizer
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return rows


def _generate_qual_model(spec: dict) -> list[dict]:
    if spec.get("runner") == "parcae":
        return _generate_parcae_model(spec)
    if spec.get("runner") == "ouro_causal_lm":
        return _generate_ouro_model(spec)
    return _generate_qwen35_model(spec)


qual_rows = []
qual_failures = []
if RUN_QUALITATIVE_GENERATION:
    specs = selected_qual_specs()
    for spec in specs:
        try:
            qual_rows.extend(_generate_qual_model(spec))
        except Exception as exc:
            print(f"[FAIL] {spec['tag']}: {type(exc).__name__}: {exc}")
            qual_failures.append({
                "family": spec.get("family", "unknown"),
                "runner": spec.get("runner", "qwen35_processor"),
                "tag": spec["tag"],
                "model_id": spec["model_id"],
                "error_type": type(exc).__name__,
                "error": str(exc),
            })
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

qual_df = pd.DataFrame(qual_rows, columns=QUAL_RESULT_COLUMNS)
qual_failures_df = pd.DataFrame(qual_failures, columns=QUAL_FAILURE_COLUMNS)
qual_df.to_csv(OUT_DIR / "bat_ball_direct_outputs.csv", index=False, encoding="utf-8-sig")
qual_failures_df.to_csv(OUT_DIR / "bat_ball_direct_failures.csv", index=False, encoding="utf-8-sig")

if not qual_failures_df.empty:
    print("Failures:")
    display(qual_failures_df)
if qual_df.empty and RUN_QUALITATIVE_GENERATION:
    raise RuntimeError("No successful qualitative outputs. Check qual_failures_df above.")

display(qual_df[[
    "family", "tag", "think", "mode", "decode", "prompt_id", "sample_idx", "label", "is_correct",
    "answer_only", "has_final_answer", "has_thinking_trace", "unexpected_thinking_trace",
    "truncated_thinking", "answer_source", "thinking_words", "leaked_thinking_words",
    "generated_tokens", "max_new_tokens", "answer",
]])

if not qual_df.empty:
    qual_model_think_summary = (
        qual_df
        .groupby(["family", "tag", "model_id", "think"], as_index=False)
        .agg(
            n=("label", "size"),
            correct_n=("is_correct", "sum"),
            lure_10_n=("label", lambda s: int((s == "lure_10").sum())),
            no_final_answer_n=("label", lambda s: int((s == "no_final_answer").sum())),
            answer_only_rate=("answer_only", "mean"),
            has_thinking_trace_n=("has_thinking_trace", "sum"),
            unexpected_thinking_n=("unexpected_thinking_trace", "sum"),
            truncated_thinking_n=("truncated_thinking", "sum"),
            avg_thinking_words=("thinking_words", "mean"),
            avg_leaked_thinking_words=("leaked_thinking_words", "mean"),
            avg_generated_tokens=("generated_tokens", "mean"),
        )
    )
    qual_model_think_summary["correct_rate"] = qual_model_think_summary["correct_n"] / qual_model_think_summary["n"]
    qual_model_think_summary["any_correct"] = qual_model_think_summary["correct_n"] > 0
    qual_model_think_summary["all_correct"] = qual_model_think_summary["correct_n"] == qual_model_think_summary["n"]
    qual_model_think_summary["correct_summary"] = (
        qual_model_think_summary["correct_n"].astype(int).astype(str)
        + "/"
        + qual_model_think_summary["n"].astype(int).astype(str)
        + " ("
        + (qual_model_think_summary["correct_rate"] * 100).round(1).astype(str)
        + "%)"
    )
    qual_correct_pivot = (
        qual_model_think_summary
        .pivot(index=["family", "tag", "model_id"], columns="think", values="correct_summary")
        .reset_index()
        .rename_axis(None, axis=1)
    )
    qual_model_think_summary.to_csv(OUT_DIR / "bat_ball_direct_model_think_summary.csv", index=False, encoding="utf-8-sig")
    qual_correct_pivot.to_csv(OUT_DIR / "bat_ball_direct_correct_pivot.csv", index=False, encoding="utf-8-sig")
    display(qual_correct_pivot)
    display(qual_model_think_summary[[
        "family", "tag", "think", "n", "correct_n", "correct_rate", "any_correct", "all_correct",
        "lure_10_n", "no_final_answer_n", "has_thinking_trace_n", "unexpected_thinking_n",
        "truncated_thinking_n", "answer_only_rate", "avg_thinking_words",
        "avg_leaked_thinking_words", "avg_generated_tokens",
    ]])

display(qual_df.groupby([
    "family", "tag", "think", "mode", "decode", "prompt_id", "label",
    "answer_only", "unexpected_thinking_trace",
]).size().reset_index(name="n"))

In [ ]:
if not qual_df.empty:
    for r in qual_df.itertuples():
        print("=" * 90)
        print(
            f"{r.family} / {r.tag} / {r.think} / {r.mode} / {r.decode} / {r.prompt_id} "
            f"/ sample={r.sample_idx} / label={r.label} / correct={r.is_correct} "
            f"/ think_trace={r.has_thinking_trace} / unexpected={r.unexpected_thinking_trace} "
            f"/ truncated={r.truncated_thinking}"
        )
        if r.thinking_excerpt:
            print("\n[thinking]")
            print(r.thinking_excerpt)
        if r.leaked_thinking_excerpt:
            print("\n[unexpected non-think reasoning]")
            print(r.leaked_thinking_excerpt)
        print("\n[answer]")
        print(r.answer)

## 2. Probe contexts


In [ ]:
BAT_BALL = (
    "A bat and a ball cost $1.10 in total. "
    "The bat costs $1.00 more than the ball. "
    "How much does the ball cost?"
)

PROBES = [
    # 답을 내기 직전: lure가 올라오는지
    {
        "id": "pre_answer_0",
        "condition": "lure_pre_answer",
        "expected_level": "high",
        "split": "discovery",
        "text": BAT_BALL + " Think briefly, then answer. The ball costs",
    },
    {
        "id": "pre_answer_1",
        "condition": "lure_pre_answer",
        "expected_level": "high",
        "split": "discovery",
        "text": BAT_BALL + " The quick answer is",
    },
    {
        "id": "pre_answer_2",
        "condition": "lure_pre_answer",
        "expected_level": "high",
        "split": "validation",
        "text": BAT_BALL + " Final answer:",
    },
    # 오답 run: 직관 오답 10 cents를 말하는 transcript
    {
        "id": "wrong_10_0",
        "condition": "wrong_10_answer",
        "expected_level": "very_high",
        "split": "discovery",
        "text": BAT_BALL + " The ball costs 10 cents. Final answer: 10 cents.",
    },
    {
        "id": "wrong_10_1",
        "condition": "wrong_10_answer",
        "expected_level": "very_high",
        "split": "discovery",
        "text": BAT_BALL + " It seems straightforward: the ball is 10 cents.",
    },
    {
        "id": "wrong_10_2",
        "condition": "wrong_10_answer",
        "expected_level": "very_high",
        "split": "validation",
        "text": BAT_BALL + " My answer is 10 cents.",
    },
    # 정답 run 초기: 처음에는 10 cents lure가 떠오르지만 아직 수정 전
    {
        "id": "correct_initial_0",
        "condition": "correct_initial_lure",
        "expected_level": "transient_high",
        "split": "discovery",
        "text": BAT_BALL + " At first glance, the ball seems to cost 10 cents, but",
    },
    {
        "id": "correct_initial_1",
        "condition": "correct_initial_lure",
        "expected_level": "transient_high",
        "split": "discovery",
        "text": BAT_BALL + " A tempting answer is 10 cents. However, checking it,",
    },
    {
        "id": "correct_initial_2",
        "condition": "correct_initial_lure",
        "expected_level": "transient_high",
        "split": "validation",
        "text": BAT_BALL + " The intuitive guess is 10 cents, but that would make",
    },
    # 정답 run 후반: 식과 검산 후 5 cents로 안정화
    {
        "id": "late_check_0",
        "condition": "correct_late_check",
        "expected_level": "low_after_check",
        "split": "discovery",
        "text": BAT_BALL + " Let x be the ball. Then x + (x + 100 cents) = 110 cents, so 2x = 10 and x = 5. Final answer: 5 cents.",
    },
    {
        "id": "late_check_1",
        "condition": "correct_late_check",
        "expected_level": "low_after_check",
        "split": "discovery",
        "text": BAT_BALL + " Check: if the ball is 5 cents, the bat is 105 cents, total 110 cents. Therefore the ball is 5 cents.",
    },
    {
        "id": "late_check_2",
        "condition": "correct_late_check",
        "expected_level": "low_after_check",
        "split": "validation",
        "text": BAT_BALL + " Solving carefully gives ball = 5 cents and bat = 105 cents. Answer: 5 cents.",
    },
    # 단순 10 cents: 숫자/토큰 자체 feature를 배제하기 위한 control
    {
        "id": "plain_10_0",
        "condition": "plain_10_sentence",
        "expected_level": "low_or_mid",
        "split": "discovery",
        "text": "The sticker on the pencil says 10 cents. There is no puzzle; the listed price is 10 cents.",
    },
    {
        "id": "plain_10_1",
        "condition": "plain_10_sentence",
        "expected_level": "low_or_mid",
        "split": "discovery",
        "text": "A parking meter displays 10 cents remaining. This sentence simply mentions 10 cents.",
    },
    {
        "id": "plain_10_2",
        "condition": "plain_10_sentence",
        "expected_level": "low_or_mid",
        "split": "validation",
        "text": "The donation jar contains a coin worth 10 cents. Nothing is being solved.",
    },
    # 함정 없는 돈 계산
    {
        "id": "money_control_0",
        "condition": "no_lure_money_control",
        "expected_level": "low",
        "split": "discovery",
        "text": "A bat costs $1.05 and a ball costs $0.05. Together they cost $1.10. How much does the ball cost? Answer: 5 cents.",
    },
    {
        "id": "money_control_1",
        "condition": "no_lure_money_control",
        "expected_level": "low",
        "split": "discovery",
        "text": "A book costs $2.15 and a toy costs $0.15. The total is $2.30. How much does the toy cost? Answer: 15 cents.",
    },
    {
        "id": "money_control_2",
        "condition": "no_lure_money_control",
        "expected_level": "low",
        "split": "validation",
        "text": "A pen costs 95 cents and an eraser costs 5 cents. What is the eraser's price? Answer: 5 cents.",
    },
]

probe_df = pd.DataFrame(PROBES)
display(probe_df[["id", "condition", "expected_level", "split", "text"]])
print(probe_df.groupby(["condition", "split"]).size().unstack(fill_value=0))


## 3. Residual capture와 Qwen-Scope projection


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=True,
).to(DEVICE).eval()

residual_by_probe = {}
for probe in tqdm(PROBES, desc="capture residuals"):
    residual_by_probe[probe["id"]] = capture_residuals(
        model,
        tokenizer,
        [probe["text"]],
        LAYERS,
        device=DEVICE,
        max_length=MAX_LENGTH,
        token_position=TOKEN_POSITION,
    )

del model, tokenizer
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("captured", len(residual_by_probe), "probe contexts")


In [ ]:
feature_matrices = {}
feature_rows = []

for layer in tqdm(LAYERS, desc="SAE projection"):
    sae = load_qwen_scope_sae(
        SAE_REPO,
        layer,
        device=DEVICE,
        dtype=DTYPE,
        top_k=SAE_TOP_K,
    )
    rows = []
    ids = []
    for probe in PROBES:
        summary = summarize_qwen_scope_features(
            residual_by_probe[probe["id"]][layer],
            sae,
            batch_size=BATCH_SIZE,
        )
        mean_vec = summary["mean"].numpy()
        max_vec = summary["max"].numpy()
        rate_vec = summary["activation_rate"].numpy()
        rows.append(mean_vec)
        ids.append(probe["id"])
        top_idx = np.argsort(mean_vec)[::-1][:10]
        for rank, feature in enumerate(top_idx, start=1):
            feature_rows.append({
                "layer": layer,
                "probe_id": probe["id"],
                "condition": probe["condition"],
                "split": probe["split"],
                "feature": int(feature),
                "rank_in_probe": rank,
                "mean_activation": float(mean_vec[feature]),
                "max_activation": float(max_vec[feature]),
                "activation_rate": float(rate_vec[feature]),
            })
    feature_matrices[layer] = {"probe_ids": ids, "matrix": np.stack(rows)}
    del sae
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

feature_rows_df = pd.DataFrame(feature_rows)
feature_rows_df.to_csv(OUT_DIR / "probe_top_features.csv", index=False, encoding="utf-8-sig")
display(feature_rows_df.head(20))


## 4. Discovery: lure 후보 feature 찾기


In [ ]:
HIGH_CONDITIONS = {"lure_pre_answer", "wrong_10_answer", "correct_initial_lure"}
LOW_CONDITIONS = {"correct_late_check", "plain_10_sentence", "no_lure_money_control"}

probe_meta = {p["id"]: p for p in PROBES}
candidate_rows = []

for layer, bundle in feature_matrices.items():
    ids = bundle["probe_ids"]
    X = bundle["matrix"]
    is_discovery = np.array([probe_meta[i]["split"] == "discovery" for i in ids])
    conds = np.array([probe_meta[i]["condition"] for i in ids])

    high = is_discovery & np.isin(conds, list(HIGH_CONDITIONS))
    low = is_discovery & np.isin(conds, list(LOW_CONDITIONS))
    wrong = is_discovery & (conds == "wrong_10_answer")
    plain10 = is_discovery & (conds == "plain_10_sentence")
    early = is_discovery & (conds == "correct_initial_lure")
    late = is_discovery & (conds == "correct_late_check")
    money = is_discovery & (conds == "no_lure_money_control")

    high_mean = X[high].mean(axis=0)
    low_mean = X[low].mean(axis=0)
    pooled = X[is_discovery].std(axis=0) + 1e-8
    discovery_effect = (high_mean - low_mean) / pooled

    wrong_minus_plain10 = X[wrong].mean(axis=0) - X[plain10].mean(axis=0)
    early_minus_late = X[early].mean(axis=0) - X[late].mean(axis=0)
    pre_minus_money = X[is_discovery & (conds == "lure_pre_answer")].mean(axis=0) - X[money].mean(axis=0)

    # 숫자 "10 cents" 자체가 아니라 CRT-lure 문맥에 특이적인 feature를 선호한다.
    score = discovery_effect + 0.5 * np.sign(wrong_minus_plain10) + 0.5 * np.sign(early_minus_late)
    top = np.argsort(score)[::-1][:DISCOVERY_TOP_N]

    for rank, feature in enumerate(top, start=1):
        candidate_rows.append({
            "layer": layer,
            "rank": rank,
            "feature": int(feature),
            "discovery_score": float(score[feature]),
            "high_mean": float(high_mean[feature]),
            "low_mean": float(low_mean[feature]),
            "discovery_effect": float(discovery_effect[feature]),
            "wrong_minus_plain10": float(wrong_minus_plain10[feature]),
            "early_minus_late": float(early_minus_late[feature]),
            "pre_minus_money": float(pre_minus_money[feature]),
        })

candidates = (
    pd.DataFrame(candidate_rows)
    .sort_values("discovery_score", ascending=False)
    .reset_index(drop=True)
)
candidates.to_csv(OUT_DIR / "candidate_lure_features_discovery.csv", index=False, encoding="utf-8-sig")
display(candidates.head(REPORT_TOP_N))


## 5. Validation: 남겨둔 paraphrase에서 기대 패턴 확인


In [ ]:
validation_rows = []

for _, cand in candidates.head(DISCOVERY_TOP_N).iterrows():
    layer = int(cand["layer"])
    feature = int(cand["feature"])
    bundle = feature_matrices[layer]
    ids = bundle["probe_ids"]
    X = bundle["matrix"]
    rows = []
    for i, pid in enumerate(ids):
        meta = probe_meta[pid]
        rows.append({
            "layer": layer,
            "feature": feature,
            "probe_id": pid,
            "condition": meta["condition"],
            "split": meta["split"],
            "activation": float(X[i, feature]),
        })
    df_feat = pd.DataFrame(rows)
    val = df_feat[df_feat["split"] == "validation"]
    means = val.groupby("condition")["activation"].mean().to_dict()

    checks = {
        "wrong_gt_plain10": means.get("wrong_10_answer", 0.0) > means.get("plain_10_sentence", 0.0),
        "wrong_gt_money": means.get("wrong_10_answer", 0.0) > means.get("no_lure_money_control", 0.0),
        "pre_gt_money": means.get("lure_pre_answer", 0.0) > means.get("no_lure_money_control", 0.0),
        "early_gt_late": means.get("correct_initial_lure", 0.0) > means.get("correct_late_check", 0.0),
        "late_le_wrong": means.get("correct_late_check", 0.0) <= means.get("wrong_10_answer", 0.0),
    }
    validation_rows.append({
        "layer": layer,
        "feature": feature,
        "discovery_rank": int(cand["rank"]),
        "discovery_score": float(cand["discovery_score"]),
        **{f"val_{k}": float(v) for k, v in means.items()},
        "pattern_score": int(sum(checks.values())),
        **checks,
    })

validation = pd.DataFrame(validation_rows).sort_values(
    ["pattern_score", "discovery_score"],
    ascending=[False, False],
)
validation.to_csv(OUT_DIR / "validation_pattern_scores.csv", index=False, encoding="utf-8-sig")
display(validation.head(REPORT_TOP_N))


In [ ]:
if validation.empty:
    raise ValueError("validation 결과가 비었습니다.")

PICK_LAYER = int(validation.iloc[0]["layer"])
PICK_FEATURE = int(validation.iloc[0]["feature"])
print("selected feature:", f"L{PICK_LAYER}/F{PICK_FEATURE}")

bundle = feature_matrices[PICK_LAYER]
plot_rows = []
for i, pid in enumerate(bundle["probe_ids"]):
    meta = probe_meta[pid]
    plot_rows.append({
        "probe_id": pid,
        "condition": meta["condition"],
        "split": meta["split"],
        "expected_level": meta["expected_level"],
        "activation": float(bundle["matrix"][i, PICK_FEATURE]),
    })
plot_df = pd.DataFrame(plot_rows)
order = [
    "lure_pre_answer",
    "wrong_10_answer",
    "correct_initial_lure",
    "correct_late_check",
    "plain_10_sentence",
    "no_lure_money_control",
]
plot_df["condition"] = pd.Categorical(plot_df["condition"], categories=order, ordered=True)
plot_df = plot_df.sort_values(["condition", "split", "probe_id"])
display(plot_df)

fig = go.Figure()
for split, sdf in plot_df.groupby("split", observed=False):
    fig.add_trace(go.Bar(
        x=[f"{r.condition}<br>{r.probe_id}" for r in sdf.itertuples()],
        y=sdf["activation"],
        name=split,
    ))
fig.update_layout(
    title=f"CRT lure MVP selected feature activation: L{PICK_LAYER}/F{PICK_FEATURE}",
    xaxis_title="condition / probe",
    yaxis_title="SAE mean activation at last token",
    template="plotly_white",
    height=520,
)
fig.show()

heat = plot_df.pivot_table(
    index="condition",
    columns="probe_id",
    values="activation",
    observed=False,
).reindex(order).fillna(0.0)
fig2 = go.Figure(data=go.Heatmap(
    z=heat.values,
    x=heat.columns.tolist(),
    y=heat.index.astype(str).tolist(),
    colorscale="Viridis",
))
fig2.update_layout(
    title=f"Heatmap for selected feature L{PICK_LAYER}/F{PICK_FEATURE}",
    template="plotly_white",
    height=420,
)
fig2.show()


## 7. H100용 Qwen/Qwen-Scope 모델군 비교

Qwen-Scope collection 기준으로 현재 feature projection에 바로 쓸 수 있는 최신 축은 Qwen3와 Qwen3.5입니다. Qwen3.6은 더 최신 모델군이지만, 이 노트북이 쓰는 Qwen-Scope SAE repo가 아직 collection에 보이지 않으므로 여기서는 제외합니다.

H100 한 장에서는 모델을 동시에 올리지 않고 **순차 로드 → projection → unload** 방식으로 실행합니다. 기본 suite는 H100에서 현실적인 검증 범위인 `qwen3_1p7b`, `qwen35_2b`, `qwen3_8b`, `qwen35_9b`입니다. 더 무겁게 보고 싶으면 `SUITE_NAME = "h100_extended"`로 바꾸세요.


In [ ]:
MODEL_SUITES = {
    "h100_recommended": [
        {
            "tag": "qwen3_1p7b_base",
            "model_id": "Qwen/Qwen3-1.7B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen35_2b_base",
            "model_id": "Qwen/Qwen3.5-2B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3.5-2B-Base-W32K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen3_8b_base",
            "model_id": "Qwen/Qwen3-8B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3-8B-Base-W64K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen35_9b_base",
            "model_id": "Qwen/Qwen3.5-9B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3.5-9B-Base-W64K-L0_50",
            "layer_policy": "quartiles",
        },
    ],
    "h100_extended": [
        {
            "tag": "qwen3_1p7b_base",
            "model_id": "Qwen/Qwen3-1.7B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen35_2b_base",
            "model_id": "Qwen/Qwen3.5-2B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3.5-2B-Base-W32K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen3_8b_base",
            "model_id": "Qwen/Qwen3-8B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3-8B-Base-W64K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen35_9b_base",
            "model_id": "Qwen/Qwen3.5-9B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3.5-9B-Base-W64K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen3_30b_a3b_base",
            "model_id": "Qwen/Qwen3-30B-A3B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3-30B-A3B-Base-W32K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen35_35b_a3b_base",
            "model_id": "Qwen/Qwen3.5-35B-A3B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3.5-35B-A3B-Base-W32K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen35_27b",
            "model_id": "Qwen/Qwen3.5-27B",
            "sae_repo": "Qwen/SAE-Res-Qwen3.5-27B-W80K-L0_50",
            "layer_policy": "quartiles",
        },
    ],
}

pd.DataFrame(
    [
        {"suite": suite, **spec}
        for suite, specs in MODEL_SUITES.items()
        for spec in specs
    ]
).drop_duplicates(["suite", "tag"]).pipe(display)


In [ ]:
def choose_probe_layers(model, policy="quartiles", explicit_layers=None):
    if explicit_layers:
        return [int(x) for x in explicit_layers]
    blocks = get_transformer_layers(model)
    n_layers = len(blocks)
    if policy == "quartiles":
        raw = [round(n_layers * q) for q in (0.25, 0.50, 0.75, 0.95)]
    elif policy == "middle_last":
        raw = [n_layers // 2, n_layers - 1]
    else:
        raise ValueError(f"unknown layer policy: {policy}")
    return sorted({min(max(int(x), 0), n_layers - 1) for x in raw})


def load_suite_model(model_id: str):
    kwargs = {
        "torch_dtype": DTYPE,
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
    }
    if DEVICE == "cuda":
        kwargs["device_map"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs).eval()
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    return model


def score_lure_candidates(local_feature_matrices, local_layers):
    rows = []
    probe_meta = {p["id"]: p for p in PROBES}
    high_conditions = {"lure_pre_answer", "wrong_10_answer", "correct_initial_lure"}
    low_conditions = {"correct_late_check", "plain_10_sentence", "no_lure_money_control"}

    for layer in local_layers:
        bundle = local_feature_matrices[layer]
        ids = bundle["probe_ids"]
        X = bundle["matrix"]
        is_discovery = np.array([probe_meta[i]["split"] == "discovery" for i in ids])
        conds = np.array([probe_meta[i]["condition"] for i in ids])

        high = is_discovery & np.isin(conds, list(high_conditions))
        low = is_discovery & np.isin(conds, list(low_conditions))
        wrong = is_discovery & (conds == "wrong_10_answer")
        plain10 = is_discovery & (conds == "plain_10_sentence")
        early = is_discovery & (conds == "correct_initial_lure")
        late = is_discovery & (conds == "correct_late_check")
        money = is_discovery & (conds == "no_lure_money_control")

        high_mean = X[high].mean(axis=0)
        low_mean = X[low].mean(axis=0)
        pooled = X[is_discovery].std(axis=0) + 1e-8
        discovery_effect = (high_mean - low_mean) / pooled
        wrong_minus_plain10 = X[wrong].mean(axis=0) - X[plain10].mean(axis=0)
        early_minus_late = X[early].mean(axis=0) - X[late].mean(axis=0)
        pre_minus_money = (
            X[is_discovery & (conds == "lure_pre_answer")].mean(axis=0)
            - X[money].mean(axis=0)
        )
        score = discovery_effect + 0.5 * np.sign(wrong_minus_plain10) + 0.5 * np.sign(early_minus_late)
        top = np.argsort(score)[::-1][:DISCOVERY_TOP_N]
        for rank, feature in enumerate(top, start=1):
            rows.append({
                "layer": int(layer),
                "rank": rank,
                "feature": int(feature),
                "discovery_score": float(score[feature]),
                "high_mean": float(high_mean[feature]),
                "low_mean": float(low_mean[feature]),
                "discovery_effect": float(discovery_effect[feature]),
                "wrong_minus_plain10": float(wrong_minus_plain10[feature]),
                "early_minus_late": float(early_minus_late[feature]),
                "pre_minus_money": float(pre_minus_money[feature]),
            })
    return (
        pd.DataFrame(rows)
        .sort_values("discovery_score", ascending=False)
        .reset_index(drop=True)
    )


def validate_lure_candidates(local_candidates, local_feature_matrices):
    probe_meta = {p["id"]: p for p in PROBES}
    rows = []
    for _, cand in local_candidates.head(DISCOVERY_TOP_N).iterrows():
        layer = int(cand["layer"])
        feature = int(cand["feature"])
        bundle = local_feature_matrices[layer]
        ids = bundle["probe_ids"]
        X = bundle["matrix"]
        feat_rows = []
        for i, pid in enumerate(ids):
            meta = probe_meta[pid]
            feat_rows.append({
                "probe_id": pid,
                "condition": meta["condition"],
                "split": meta["split"],
                "activation": float(X[i, feature]),
            })
        df_feat = pd.DataFrame(feat_rows)
        val = df_feat[df_feat["split"] == "validation"]
        means = val.groupby("condition")["activation"].mean().to_dict()
        checks = {
            "wrong_gt_plain10": means.get("wrong_10_answer", 0.0) > means.get("plain_10_sentence", 0.0),
            "wrong_gt_money": means.get("wrong_10_answer", 0.0) > means.get("no_lure_money_control", 0.0),
            "pre_gt_money": means.get("lure_pre_answer", 0.0) > means.get("no_lure_money_control", 0.0),
            "early_gt_late": means.get("correct_initial_lure", 0.0) > means.get("correct_late_check", 0.0),
            "late_le_wrong": means.get("correct_late_check", 0.0) <= means.get("wrong_10_answer", 0.0),
        }
        rows.append({
            "layer": layer,
            "feature": feature,
            "discovery_rank": int(cand["rank"]),
            "discovery_score": float(cand["discovery_score"]),
            **{f"val_{k}": float(v) for k, v in means.items()},
            "pattern_score": int(sum(checks.values())),
            **checks,
        })
    return pd.DataFrame(rows).sort_values(
        ["pattern_score", "discovery_score"],
        ascending=[False, False],
    )


In [ ]:
def run_suite_model(spec: dict) -> dict:
    tag = spec["tag"]
    model_id = spec["model_id"]
    sae_repo = spec["sae_repo"]
    model_out = OUT_DIR / "model_suite" / tag
    model_out.mkdir(parents=True, exist_ok=True)

    print(f"\n===== {tag} =====")
    print("model:", model_id)
    print("sae:", sae_repo)

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = load_suite_model(model_id)
    layers = choose_probe_layers(
        model,
        policy=spec.get("layer_policy", "quartiles"),
        explicit_layers=spec.get("layers"),
    )
    print("layers:", layers)

    residual_by_probe = {}
    try:
        for probe in tqdm(PROBES, desc=f"{tag}: capture"):
            residual_by_probe[probe["id"]] = capture_residuals(
                model,
                tokenizer,
                [probe["text"]],
                layers,
                device=DEVICE,
                max_length=MAX_LENGTH,
                token_position=TOKEN_POSITION,
            )
    finally:
        del model, tokenizer
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    local_feature_matrices = {}
    for layer in tqdm(layers, desc=f"{tag}: SAE projection"):
        sae = load_qwen_scope_sae(
            sae_repo,
            layer,
            device=DEVICE,
            dtype=DTYPE,
            top_k=SAE_TOP_K,
        )
        rows = []
        ids = []
        for probe in PROBES:
            summary = summarize_qwen_scope_features(
                residual_by_probe[probe["id"]][layer],
                sae,
                batch_size=BATCH_SIZE,
            )
            rows.append(summary["mean"].numpy())
            ids.append(probe["id"])
        local_feature_matrices[layer] = {"probe_ids": ids, "matrix": np.stack(rows)}
        del sae
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    del residual_by_probe
    gc.collect()

    local_candidates = score_lure_candidates(local_feature_matrices, layers)
    local_validation = validate_lure_candidates(local_candidates, local_feature_matrices)
    local_candidates.to_csv(model_out / "candidate_lure_features.csv", index=False, encoding="utf-8-sig")
    local_validation.to_csv(model_out / "validation_pattern_scores.csv", index=False, encoding="utf-8-sig")

    best = local_validation.iloc[0].to_dict()
    best.update({
        "tag": tag,
        "model_id": model_id,
        "sae_repo": sae_repo,
        "layers": ",".join(map(str, layers)),
        "output_dir": str(model_out),
    })
    print("best:", {k: best[k] for k in ["tag", "layer", "feature", "pattern_score", "discovery_score"]})
    return best


In [ ]:
# H100에서 실행할 때 True로 바꾸세요.
RUN_MODEL_SUITE = False
SUITE_NAME = "h100_recommended"  # "h100_recommended" | "h100_extended"
MAX_MODELS = None  # 예: 2 로 두면 앞의 2개만 smoke test

suite_results = []
if RUN_MODEL_SUITE:
    specs = MODEL_SUITES[SUITE_NAME]
    if MAX_MODELS is not None:
        specs = specs[: int(MAX_MODELS)]
    for spec in specs:
        try:
            suite_results.append(run_suite_model(spec))
        except Exception as exc:
            print(f"[SKIP/FAIL] {spec['tag']}: {type(exc).__name__}: {exc}")
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

    suite_df = pd.DataFrame(suite_results)
    suite_df.to_csv(OUT_DIR / f"{SUITE_NAME}_summary.csv", index=False, encoding="utf-8-sig")
    display(suite_df.sort_values(["pattern_score", "discovery_score"], ascending=[False, False]))
else:
    print("RUN_MODEL_SUITE=False 입니다. H100에서 모델군 비교를 돌릴 때 True로 바꾸세요.")


## 8. 해석 기준

MVP에서 가장 먼저 볼 것은 `pattern_score`와 선택 feature의 막대그래프입니다.

- 좋은 후보: `wrong_10_answer`, `lure_pre_answer`, `correct_initial_lure`가 높고 `correct_late_check`, `plain_10_sentence`, `no_lure_money_control`이 낮습니다.
- 애매한 후보: `plain_10_sentence`도 같이 높으면 “10 cents 토큰/가격 문맥” feature일 가능성이 큽니다.
- 실패한 후보: `no_lure_money_control`이 높으면 CRT lure가 아니라 일반 돈 계산 feature일 수 있습니다.

이 MVP가 통과하면 다음 단계는 실제 Qwen3 `non_think`/`think` generation run을 수집하고, wrong/correct run의 token trajectory에 같은 feature가 재현되는지 보는 것입니다.